# Discovery - BuildingRocket

## Rocket Class
(Last updated: July 7, 2026)

#### Rocketpy imports:

In [ ]:
from rocketpy import (
    Environment,
    Rocket,
    Flight,
)
import datetime
from rocketpy.motors import Fluid, CylindricalTank, MassFlowRateBasedTank, LiquidMotor

#### Defining environment:

In [ ]:
flight_date = datetime.date(2026, 8, 15)
env = Environment(latitude=28.6, longitude=-80.6, elevation=195)

# env.set_date((flight_date.year, flight_date.month, flight_date.day, 12))
# env.set_atmospheric_model(type="Forecast", file="GFS")
env.set_atmospheric_model("custom_atmosphere", wind_u=8.3333, wind_v=0)


#### Defining main parameters:
- Radius (m)
- Mass without motors (kg)
- Moment of inertia (using formula of a solid cylinder)
- Drag coefficient as a function of Mach number when motor is off/on
- Position of center of mass without motors (m)
- Orientation of rocket

In [ ]:
rocket = Rocket(
    radius=15.2 / 200,
    mass=34.954,  # rocket's mass without the motor and propellant in kg
    inertia=( 
        56.7, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        56.7, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        0.101, # (1/2) * mass * radius^2
    ),  # in relation to the rocket's center of mass without motor
    power_off_drag="DragCurve.csv",
    power_on_drag="DragCurve.csv",
    center_of_mass_without_motor=2.22,
    coordinate_system_orientation="tail_to_nose",
)

#### Configuring the motor: (WIP)
- Fluids
- Tank geometry

In [ ]:
oxidizer = Fluid(name="N2O", density=1220)
fuel = Fluid(name="ethanol", density=789)

shape_oxidizer_tank = CylindricalTank(radius=0.074, height=0.64)
shape_fuel_tank = CylindricalTank(radius=0.074, height=0.254)

Tank parameters for oxidizer and fuel tanks:
- Liquid (refer to variable name created for fluid)
- Geometry (refer to variable name created for tanks)
- Flux time (total exit time for fluids) (s)
- Initial liquid mass (kg)
- Liquid mass flowing in (function)
- Liquid mass flowing out (function)*

*based off mass of motor vs. time data from OpenRocket

In [ ]:
oxidizer_tank = MassFlowRateBasedTank(
    name="oxidizer tank",
    liquid=oxidizer,
    geometry=shape_oxidizer_tank,
    flux_time=4,
    initial_liquid_mass=7.010,
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=lambda t: 1.753 * t,
)

fuel_tank = MassFlowRateBasedTank(
    name="fuel tank",
    liquid=fuel,
    geometry=shape_fuel_tank,
    flux_time=4,
    initial_liquid_mass=2.420,
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=lambda t: 0.605 * t,
)

#### Motor parameters (WIP)
- Thrust source (.csv file)
- Dry_mass (total mass of NON-dynamic mass elements) (kg)
- Center of dry mass position (m)
- Dry inertia
- Nozzle position (m)
- Nozzle radius (m)
- Burn time (s)

In [ ]:
motor = LiquidMotor(
    thrust_source="thrust.csv",
    dry_mass=9.780, # referring to "Engine Section" of Discovery's mass distribution spreadsheet
    center_of_dry_mass_position=0.1953, # UPDATE
    # dry_inertia=( , , ), # UPDATE
    nozzle_position=0,
    nozzle_radius=0.06,
    burn_time=4.56,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

motor.add_tank(tank=oxidizer_tank, position=0.8904)
motor.add_tank(tank=fuel_tank, position=1.8274)

#### Adding the rocket's components:
- Rail buttons
- Nose cone
- Fins
- Parachutes

In [ ]:
rail_buttons = rocket.set_rail_buttons(
    upper_button_position=0.0818,
    lower_button_position=0.6182,
    angular_position=45,
)

nose_cone = rocket.add_nose(length=0.451, kind="ogive", position=4.41)

fin_set = rocket.add_trapezoidal_fins(
    n=3,
    root_chord=0.298,
    tip_chord=0.139,
    span=0.152, # height of a single fin
    cant_angle=0,
    sweep_length=0.14,
    airfoil=("Airfoil.csv", "radians"),
    position=0.298,)

main = rocket.add_parachute(
    name="main",
    cd_s=16.074, # reference area * drag coefficient
    trigger=610,  # ejection altitude in meters
    sampling_rate=105,
)

drogue = rocket.add_parachute(
    name="drogue",
    cd_s=0.5905, # reference area * drag coefficient
    trigger="apogee",  # ejection altitude in meters
    sampling_rate=105,
    lag=2,
)

#### Flight parameters:

In [ ]:
flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=9.21,
    inclination=85.0,
    heading=270.0,
)